# 02 · Controlled head-to-head comparison with re-implemented baselines

Reviewer 1 (comments 4 and 10) and Reviewer 2 (comment 4) asked for a comparison under a common protocol. Every model below is therefore trained and evaluated:
* on **identical partitions**: official train/test files, the same contiguous validation block, and the same chronological CICIDS-2018 subset;
* with **identical preprocessing** (one-hot + Min–Max fitted on the training split) and **class-weighted loss** (no resampling for any model);
* on **identical test targets** (the final events of the TA-LSTMAE test windows), with the same threshold (0.5) and the same metrics engine, including **causal MTTD for every model**;
* with the **same seeds** (`cfg.seeds`), reporting mean ± SD, 95% CIs and paired tests.

| Model | Role |
|---|---|
| TA-LSTMAE (TAE + HBLT, full) | proposed |
| TA-LSTMAE causal (TAE + unidirectional LSTM) | simplified, strictly causal candidate (R1-5) |
| TA-LSTMAE simplified (TAE + CNN only) | smallest variant (R1-5) |
| HAE-HRL (re-implemented) | strongest baseline in the original comparison scope |
| ADFCNN-BiLSTM (re-implemented) | representative recent (2025) attention / deformable-CNN / BiLSTM method (R1-10) |

Unreported baseline hyperparameters are listed in the next cell and must be stated in the manuscript. Run notebook 01 first only if you want the shared outputs folder; this notebook is self-contained.

In [1]:
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
ROOT = Path.cwd() if (Path.cwd() / "talstmae").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import tensorflow as tf
from talstmae import Config, set_all_seeds
from talstmae import data as DT, models as M, metrics as MT, experiment as X, alerts as AL

SMOKE_TEST = False      # True = tiny synthetic data, ONLY to check that the code executes. Never report its numbers.
RESUME = True           # continue an interrupted run (finished models/seeds are skipped)
cfg = Config().smoke() if SMOKE_TEST else Config()
if os.environ.get("TALSTMAE_SETTINGS"):          # settings written by 00_RUN_ME.ipynb override the lines above
    _s = json.loads(Path(os.environ["TALSTMAE_SETTINGS"]).read_text())
    _base = Config(**{k: (tuple(v) if k in ("datasets", "seeds") else v) for k, v in _s.get("config", {}).items()})
    SMOKE_TEST, RESUME = bool(_s.get("smoke_test")), bool(_s.get("resume", True))
    cfg = _base.smoke() if SMOKE_TEST else _base
    print("settings from 00_RUN_ME:", _s)
if SMOKE_TEST:
    print("=" * 78 + "\nSMOKE TEST MODE: synthetic placeholder data. Results are INVALID and must not be reported.\n" + "=" * 78)
set_all_seeds(cfg.seed, cfg.deterministic_ops)
print("TensorFlow", tf.__version__, "| outputs ->", cfg.out.resolve())

from talstmae import baselines as B
for k, v in B.BASELINE_NOTES.items():
    print(f"• {k}: {v}\n")

I0000 00:00:1794275156.341065    3705 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1794275157.080972    3705 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


settings from 00_RUN_ME: {'smoke_test': False, 'resume': True, 'latency_events': None, 'config': {'datasets': ['nsl_kdd', 'unsw_nb15', 'cicids2018'], 'seeds': [42, 43, 44]}}
TensorFlow 2.21.0 | outputs -> /tmp/claude-0/-home-claude/5b43f487-a08c-5b08-be88-1e2ff98cfe79/scratchpad/srv/ta-lstmae/outputs
• HAE-HRL (re-implemented): Xue, Kang and Yu, Comput. Secur. 151 (2025) 104328. Re-implemented from the published description: HAE = CNN-GRU autoencoder with dual (max + average) pooling, latent size matched to TA-LSTMAE (16/23/16); HRL = 1D-CNN stem + two residual blocks + LSTM classifier. Filter counts (32/64), LSTM units (64), Dense(64) head and dropout 0.2 are assumptions where the paper is not explicit. SMOTE+RUS of the original are replaced by class weighting (common protocol).

• ADFCNN-BiLSTM (re-implemented): Li, Li and Jia, Sensors 25(5) (2025) 1382. Two blocks of 1D deformable convolution (128 and 256 channels, kernel 3) + ReLU + MaxPool(4) + BatchNorm + ECA, spatial attention m

In [2]:
DATA = {ds: DT.prepare_dataset(ds, cfg) for ds in cfg.datasets}

[nsl_kdd]


  NSL-KDD train                  125,973  (expected 125,973)  OK
  NSL-KDD test                    22,544  (expected 22,544)  OK


  input_dim=122 OK | train=113,376 val=12,597 test=22,544 | class_weight={0: 0.9356, 1: 1.0739} | timestamps: synthetic exponential (mean=0.1s, seeds=1001/1501), resolution=Nones
[unsw_nb15]


  UNSW-NB15 train                175,341  (expected 175,341)  OK
  UNSW-NB15 test                  82,332  (expected 82,332)  OK
  note: UNSW-NB15 records permuted (seed 2024); label runs train 8,172 -> 76,562, test 5 -> 41,083


  input_dim=194 OK | train=157,807 val=17,534 test=82,332 | class_weight={0: 1.5642, 1: 0.7349} | timestamps: synthetic exponential (mean=0.1s, seeds=1002/1502), resolution=1.0s
[cicids2018]


  CICIDS-2018 attack categories: ['Benign', 'FTP-BruteForce', 'SSH-Bruteforce']
  CICIDS-2018 subset (1/6)       174,763  (expected 174,763)  OK


  input_dim=78 OK | train=110,102 val=12,233 test=52,428 | class_weight={0: 0.7375, 1: 1.5526} | timestamps: native flow timestamps (chronological split), resolution=1.0s


## 1. Train all models × seeds × datasets
Run time is roughly 5 models × 5 seeds × 3 datasets. On one modern GPU, expect several hours.

In [3]:
RUNS = X.run_head_to_head(DATA, cfg, models=X.HEAD_TO_HEAD_MODELS, seeds=cfg.seeds, resume=RESUME)
RUNS.head()

[2026-11-10 08:46:09] [1/45] nsl_kdd seed=42 TA-LSTMAE (TAE + HBLT, full): already done, skipped (resume)


[2026-11-10 08:46:09] [2/45] nsl_kdd seed=42 TA-LSTMAE causal (TAE + unidirectional LSTM): already done, skipped (resume)


[2026-11-10 08:46:09] [3/45] nsl_kdd seed=42 TA-LSTMAE simplified (TAE + CNN only): already done, skipped (resume)


[2026-11-10 08:46:09] [4/45] nsl_kdd seed=42 HAE-HRL (re-implemented): already done, skipped (resume)


[2026-11-10 08:46:09] [5/45] nsl_kdd seed=42 ADFCNN-BiLSTM (re-implemented): already done, skipped (resume)


[2026-11-10 08:46:09] [6/45] nsl_kdd seed=43 TA-LSTMAE (TAE + HBLT, full): already done, skipped (resume)


... (36 outputs left out)

[2026-11-10 08:46:10] [43/45] cicids2018 seed=44 TA-LSTMAE simplified (TAE + CNN only): already done, skipped (resume)


[2026-11-10 08:46:10] [44/45] cicids2018 seed=44 HAE-HRL (re-implemented): already done, skipped (resume)


[2026-11-10 08:46:10] [45/45] cicids2018 seed=44 ADFCNN-BiLSTM (re-implemented): already done, skipped (resume)


,dataset,model,seed,params,train_seconds,accuracy,precision,recall,f1,TN,FP,FN,TP,n,MTTD_causal,pMTTR_causal,MTTD_symmetric,APR,DetectionRate_causal,DetectionRate_symmetric
0,nsl_kdd,"TA-LSTMAE (TAE + HBLT, full)",42,267435,437.2,0.784646,0.930569,0.671812,0.780298,9064,643,4210,8618,22535,0.066060,30.066060,0.005606,0.930569,0.975133,0.998285
1,nsl_kdd,TA-LSTMAE causal (TAE + unidirectional LSTM),42,140235,196.7,0.757089,0.915574,0.631509,0.747463,8960,747,4727,8101,22535,0.075852,30.075852,0.007123,0.915574,0.970299,0.998207
2,nsl_kdd,TA-LSTMAE simplified (TAE + CNN only),42,116651,187.7,0.783980,0.933929,0.667758,0.778727,9101,606,4262,8566,22535,0.067400,30.067400,0.005803,0.933929,0.973418,0.997895
3,nsl_kdd,HAE-HRL (re-implemented),42,135435,447.2,0.799246,0.967147,0.670097,0.791674,9415,292,4232,8596,22535,0.066184,30.066184,0.005892,0.967147,0.970455,0.996648
4,nsl_kdd,ADFCNN-BiLSTM (re-implemented),42,340773,1010.9,0.750033,0.913648,0.619426,0.738304,8956,751,4882,7946,22535,0.079469,30.079469,0.008851,0.913648,0.964375,0.996726


## 2. Summary: mean ± SD and 95% confidence intervals

In [4]:
SUM = MT.summarize_runs(RUNS)
SUM.to_csv(cfg.out / "head_to_head_summary.csv", index=False)
TAB = MT.format_mean_sd(SUM, ("accuracy", "precision", "recall", "f1", "MTTD_causal"))
TAB["params"] = RUNS.groupby(["dataset", "model"], sort=False)["params"].first().values
TAB["train_s_mean"] = RUNS.groupby(["dataset", "model"], sort=False)["train_seconds"].mean().round(0).values
TAB.to_csv(cfg.out / "tables_head_to_head.csv", index=False)
TAB

,dataset,model,runs,accuracy,precision,recall,f1,MTTD_causal,params,train_s_mean
0,nsl_kdd,"TA-LSTMAE (TAE + HBLT, full)",3,0.7824 ± 0.0030,0.9387 ± 0.0115,0.6610 ± 0.0098,0.7756 ± 0.0042,0.0690 ± 0.0028,267435,459.0
1,nsl_kdd,TA-LSTMAE causal (TAE + unidirectional LSTM),3,0.7564 ± 0.0180,0.9286 ± 0.0252,0.6199 ± 0.0198,0.7434 ± 0.0198,0.0785 ± 0.0050,140235,193.0
2,nsl_kdd,TA-LSTMAE simplified (TAE + CNN only),3,0.7825 ± 0.0140,0.9333 ± 0.0160,0.6654 ± 0.0135,0.7769 ± 0.0147,0.0682 ± 0.0037,116651,187.0
3,nsl_kdd,HAE-HRL (re-implemented),3,0.7877 ± 0.0100,0.9316 ± 0.0327,0.6777 ± 0.0136,0.7843 ± 0.0075,0.0650 ± 0.0026,135435,586.0
4,nsl_kdd,ADFCNN-BiLSTM (re-implemented),3,0.7496 ± 0.0078,0.9263 ± 0.0148,0.6087 ± 0.0118,0.7346 ± 0.0090,0.0821 ± 0.0030,340773,931.0
5,unsw_nb15,"TA-LSTMAE (TAE + HBLT, full)",3,0.8550 ± 0.0094,0.8300 ± 0.0211,0.9274 ± 0.0159,0.8757 ± 0.0053,0.0132 ± 0.0030,310426,504.0
6,unsw_nb15,TA-LSTMAE causal (TAE + unidirectional LSTM),3,0.8318 ± 0.0146,0.7887 ± 0.0339,0.9527 ± 0.0365,0.8620 ± 0.0059,0.0086 ± 0.0066,179642,200.0
7,unsw_nb15,TA-LSTMAE simplified (TAE + CNN only),3,0.8594 ± 0.0100,0.8459 ± 0.0326,0.9129 ± 0.0306,0.8774 ± 0.0046,0.0162 ± 0.0059,155610,194.0
8,unsw_nb15,HAE-HRL (re-implemented),3,0.7928 ± 0.0321,0.7299 ± 0.0281,0.9919 ± 0.0079,0.8408 ± 0.0216,0.0013 ± 0.0011,141018,994.0
9,unsw_nb15,ADFCNN-BiLSTM (re-implemented),3,0.8791 ± 0.0103,0.8311 ± 0.0168,0.9799 ± 0.0062,0.8993 ± 0.0072,0.0035 ± 0.0011,340773,1834.0


## 3. Significance
* **Across seeds:** a paired t-test (and Wilcoxon, when n ≥ 5) on F1 between the proposed model and every other model, with Holm adjustment.
* **Within seed 42:** an exact McNemar test on the paired test-set predictions.

In [5]:
tests = []
for ds in DATA:
    for other in X.HEAD_TO_HEAD_MODELS[1:]:
        for metric in ("f1", "accuracy"):
            tests.append(MT.paired_seed_test(RUNS, metric, X.TA_FULL, other, ds))
TESTS = pd.DataFrame(tests)
ok = TESTS.paired_t_p.notna()
TESTS["paired_t_p_holm"] = np.nan
TESTS.loc[ok, "paired_t_p_holm"] = MT.holm(TESTS.loc[ok, "paired_t_p"].values)
mc = []
for ds in DATA:
    a = X.load_run_predictions(cfg, ds, X.TA_FULL, cfg.seeds[0])
    for other in X.HEAD_TO_HEAD_MODELS[1:]:
        b = X.load_run_predictions(cfg, ds, other, cfg.seeds[0])
        mc.append(dict(dataset=ds, model_b=other, mcnemar_p=MT.mcnemar_exact(
            a["y_true"], (a["prob_test"] >= cfg.threshold).astype(int), (b["prob_test"] >= cfg.threshold).astype(int))))
TESTS.to_csv(cfg.out / "head_to_head_paired_tests.csv", index=False)
pd.DataFrame(mc).to_csv(cfg.out / "head_to_head_mcnemar_seed42.csv", index=False)
display(TESTS.round(4)); display(pd.DataFrame(mc).round(4))

,dataset,metric,model_a,model_b,n_seeds,mean_diff,paired_t_p,wilcoxon_p,paired_t_p_holm
0,nsl_kdd,f1,"TA-LSTMAE (TAE + HBLT, full)",TA-LSTMAE causal (TAE + unidirectional LSTM),3,0.0323,0.1134,NaN,1.0000
1,nsl_kdd,accuracy,"TA-LSTMAE (TAE + HBLT, full)",TA-LSTMAE causal (TAE + unidirectional LSTM),3,0.0259,0.1584,NaN,1.0000
2,nsl_kdd,f1,"TA-LSTMAE (TAE + HBLT, full)",TA-LSTMAE simplified (TAE + CNN only),3,-0.0013,0.9017,NaN,1.0000
3,nsl_kdd,accuracy,"TA-LSTMAE (TAE + HBLT, full)",TA-LSTMAE simplified (TAE + CNN only),3,-0.0001,0.9900,NaN,1.0000
4,nsl_kdd,f1,"TA-LSTMAE (TAE + HBLT, full)",HAE-HRL (re-implemented),3,-0.0086,0.0548,NaN,1.0000
5,nsl_kdd,accuracy,"TA-LSTMAE (TAE + HBLT, full)",HAE-HRL (re-implemented),3,-0.0053,0.3725,NaN,1.0000
6,nsl_kdd,f1,"TA-LSTMAE (TAE + HBLT, full)",ADFCNN-BiLSTM (re-implemented),3,0.0411,0.0106,NaN,0.2322
7,nsl_kdd,accuracy,"TA-LSTMAE (TAE + HBLT, full)",ADFCNN-BiLSTM (re-implemented),3,0.0327,0.0099,NaN,0.2281
8,unsw_nb15,f1,"TA-LSTMAE (TAE + HBLT, full)",TA-LSTMAE causal (TAE + unidirectional LSTM),3,0.0137,0.0020,NaN,0.0469
9,unsw_nb15,accuracy,"TA-LSTMAE (TAE + HBLT, full)",TA-LSTMAE causal (TAE + unidirectional LSTM),3,0.0231,0.0184,NaN,0.3856


,dataset,model_b,mcnemar_p
0,nsl_kdd,TA-LSTMAE causal (TAE + unidirectional LSTM),0.0000
1,nsl_kdd,TA-LSTMAE simplified (TAE + CNN only),0.6435
2,nsl_kdd,HAE-HRL (re-implemented),0.0000
3,nsl_kdd,ADFCNN-BiLSTM (re-implemented),0.0000
4,unsw_nb15,TA-LSTMAE causal (TAE + unidirectional LSTM),0.0000
5,unsw_nb15,TA-LSTMAE simplified (TAE + CNN only),0.0000
6,unsw_nb15,HAE-HRL (re-implemented),0.0000
7,unsw_nb15,ADFCNN-BiLSTM (re-implemented),0.0000
8,cicids2018,TA-LSTMAE causal (TAE + unidirectional LSTM),0.0001
9,cicids2018,TA-LSTMAE simplified (TAE + CNN only),0.3742


## 4. Reading the result
* If the proposed model is not significantly better than a re-implemented baseline, the manuscript should claim **parity**, not superiority. The contribution then rests on the operational-metrics layer (R1-10, R2-1).
* If the causal or CNN-only variant is statistically indistinguishable from the full model, the simpler model should be adopted or recommended (R1-5).
* Because MTTD is now measured for **every** model with the same engine and timestamps, Table 5 can replace "N/A" with measured values for the re-implemented baselines.